# HIERARCHY with Claude Fable 5.2 – the experiment

**This notebook is only about HIERARCHY.** The vegetable-market scenario of Win as much as you can has its own notebook, `PRICE_COMMITTEE_Fable_vs_Haiku.ipynb`.

Can a stronger model solve HIERARCHY with less help than Claude Haiku needed? Steps 1–4 use **card deal 1**, the same cards as games 10 and 12–16, where the answer key needs 14 rounds.

| Step | What is tested | Question |
|---|---|---|
| 1 | Everyone knows the goal, 5 rounds | Does Fable answer properly, with no errors or cut-off answers? |
| 2 | Everyone knows the goal, full game | Can Fable organise itself without being told how? |
| 3 | Only A knows the goal, full game | Does A pass on the goal and take the lead? |
| 4 | The procedures, full games | Only if Fable got stuck in step 2: how many patches does it need? |
| 5 | Steps 2 and 3 with card deal 2 | Does the result hold with other cards? |
| 6 | Mixed organisations | Does Fable’s coordination survive working with a weaker model – as leaders, or as employees? |

**Cost.** Fable 5.2 costs \$8 per million input tokens and \$40 per million output tokens through the API. Steps 2 and 3 cost about \$6–8 each with card deal 1; expect about the same in step 5, and less for Fable’s share in step 6. Set a spending limit in the Anthropic Console. After each game, the notebook prints its estimated cost.

Run the setup cells first. After that you can run any step on its own; you do not need to change anything.

## Get the code, install and add the key

In [ ]:
import os, glob, subprocess
from google.colab import files

REPO = "https://github.com/YetiKetil/hierarchy_agents"
FOLDER = "/content/hierarchy_agents"
if os.path.exists(FOLDER):
    subprocess.run(["git", "-C", FOLDER, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, FOLDER], check=True)
os.chdir(FOLDER)
print("Working in:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Reads ANTHROPIC_API_KEY from Colab's Secrets panel (the key icon on the left; switch on Notebook access)
from google.colab import userdata
try:
    value = userdata.get("ANTHROPIC_API_KEY") or ""
except Exception:
    value = ""
lines = [l.strip() for l in value.splitlines() if l.strip()]
if lines:
    os.environ["ANTHROPIC_API_KEY"] = lines[0]
    print("ANTHROPIC_API_KEY found")
    if len(lines) > 1:
        print("  Warning: the secret has more than one line. Only the first line is used.")
else:
    print("ANTHROPIC_API_KEY not set – add it in Secrets and run this cell again")

## Settings and helpers

Run this cell once. It sets the models, the card deal and a generous answer limit, and defines `report`, which prints each result next to the earlier results with the same cards.

In [ ]:
# ---- settings
MODEL = "claude-fable-5-2"
SMALL = "claude-haiku-5-5"   # the other model in the mixed organisations
SEED = 1                     # card deal 1: the same cards as games 10 and 12-16
CONCURRENT = 3               # simultaneous API calls; lower to 1 if you see rate-limit errors
MAX_TOKENS = 16000           # room for long answers, so nothing is cut off (you only pay for what is used)
PRICE_IN, PRICE_OUT = 8, 40  # Fable 5.2, dollars per million tokens
PREFIX = ""                  # used only when testing the notebook

import os, glob, json

REFERENCE = {   # earlier results with the same cards
    ("alle_kjenner_maalet", 1): "Claude Haiku (game 6): standstill after 32 rounds, 11 cards short.",
    ("alle_kjenner_maalet", 2): "Claude Haiku (game 7): standstill after 27 rounds, 12 cards short.",
    ("grunnbetingelse", 1): "Claude Haiku (game 1): standstill after 18 rounds, 15 cards short.",
    ("grunnbetingelse", 2): "No earlier game with these cards.",
    ("prosedyre", 1): "Claude Haiku (game 10): time limit after 60 rounds, 4 cards short.",
    ("prosedyre_fasit_regler", 1): "Claude Haiku (game 15): solved in round 39. Gemini Flash-Lite (game 16): 10 cards short.",
    ("mixed_fable_leaders", 1): "Same cards, one model throughout: Haiku got stuck (32 rounds, 11 short); Fable solved it in round 20.",
    ("mixed_fable_employees", 1): "Same cards, one model throughout: Haiku got stuck (32 rounds, 11 short); Fable solved it in round 20.",
}
ANSWER_KEY = {1: 14, 2: 16}
OUTCOME = {"løst": "SOLVED", "stillstand": "standstill", "tidsgrense": "time limit reached", "api_feil": "stopped by API errors"}


# ------------------------------------------------------------------ HIERARCHY
def prepare(condition):
    """A small condition file that inherits from `condition` with a higher answer limit."""
    path = f"betingelser/en/_fable_{condition}.yaml"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"arver: {condition}.yaml\nnavn: {condition}_en\nmaks_tokens: {MAX_TOKENS}\n")
    return path


def prepare_mixed(name, leaders, employees, base="alle_kjenner_maalet"):
    """A mixed organisation: A, B and C use `leaders`, D-G use `employees`."""
    path = f"betingelser/en/_{name}.yaml"
    roles = "".join(f"  {r}:\n    modell: {leaders}\n" for r in "ABC")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"arver: {base}.yaml\nnavn: {name}_en\nmodell: {employees}\nleverandor: null\n"
                f"maks_tokens: {MAX_TOKENS}\nroller:\n{roles}")
    return path


def _models(L):
    cfg = L["konfig"]
    return {r: ((cfg.get("roller") or {}).get(r) or {}).get("modell", cfg["modell"]) for r in "ABCDEFG"}


def _cost(L):
    """Estimated cost of the Fable part of a game."""
    models = _models(L)
    inn = sum(t.get("inn", 0) for r, t in L.get("tokens", {}).items() if models.get(r) == MODEL)
    ut = sum(t.get("ut", 0) for r, t in L.get("tokens", {}).items() if models.get(r) == MODEL)
    return (inn * PRICE_IN + ut * PRICE_OUT) / 1e6


def _finish_order(L):
    hands = [r["hender_start"] for r in L["runder"]] + [L["slutt_hender"]]
    done = {}
    for i, h in enumerate(hands):
        for role, cards in h.items():
            if role not in done and len(cards) == 4 and len(set(cards)) == 1:
                done[role] = i
    return ", ".join(f"{r} {n}" for r, n in sorted(done.items(), key=lambda x: (x[1], x[0]))) or "none"


def _uses_fable(L):
    return MODEL in _models(L).values() and L.get("modus") != "test" or PREFIX


def report(condition, seed=SEED):
    """Prints the result of the latest game of `condition` with these cards that used Fable."""
    folder = f"logger/{PREFIX}{condition}_en"
    logs = []
    for p in glob.glob(f"{folder}/*.json"):
        L = json.load(open(p, encoding="utf-8"))
        if L["seed"] == seed and _uses_fable(L):
            logs.append((os.path.getmtime(p), L))
    if not logs:
        print(f"No game with {MODEL} and card deal {seed} found in {folder}.")
        return
    L = max(logs, key=lambda x: x[0])[1]
    m, t, models = L["maalinger"], L.get("tokens_sum", {}), _models(L)
    print(f"RESULT: {OUTCOME.get(m['utfall'], m['utfall'])} after {m['runder']} rounds   (card deal {seed})")
    if len(set(models.values())) > 1:
        print(f"  Models: A–C {models['A']}, D–G {models['D']}")
    print(f"  Cards missing:   {m['uorden_start']} at the start, {m['uorden_slutt']} at the end")
    print(f"  Finished (round): {_finish_order(L)}")
    print(f"  Swaps: {m['bytter']}   Messages: {m['meldinger']}   Share of messages via A: {m['toppleder_andel']:.0%}")
    print(f"  API calls: {t.get('kall', 0)}   API errors: {t.get('feil', 0)}   Truncated answers: {t.get('avkuttet', 0)}")
    note = " (Fable's share only)" if len(set(models.values())) > 1 else ""
    print(f"  Estimated cost: ${_cost(L):.2f}{note}")
    if t.get("feil") or t.get("avkuttet"):
        print("  ! Check before going on: some answers failed or were cut off.")
    print(f"\nFor comparison – {REFERENCE.get((condition, seed), '')}")
    print(f"The answer key (an algorithm with full overview) solves these cards in {ANSWER_KEY.get(seed, '?')} rounds.")


# ------------------------------------------------------------------ overview
def fable_table():
    rows, total = [], 0.0
    for p in sorted(glob.glob("logger/*/*.json"), key=os.path.getmtime):
        L = json.load(open(p, encoding="utf-8"))
        if not _uses_fable(L):
            continue
        m, c, models = L["maalinger"], _cost(L), _models(L)
        who = "Fable" if set(models.values()) == {MODEL} else "mixed"
        total += c
        rows.append((L["betingelse"], L["seed"], who, OUTCOME.get(m["utfall"], m["utfall"]), m["runder"], m["uorden_slutt"], c))
    if not rows:
        print(f"No games with {MODEL} yet.")
        return
    print(f"{'condition':32} {'deal':>4} {'models':>6}  {'outcome':22} {'rounds':>6} {'missing':>8} {'cost':>8}")
    for r in rows:
        print(f"{r[0]:32} {r[1]:>4} {r[2]:>6}  {r[3]:22} {r[4]:>6} {r[5]:>8} {'$' + format(r[6], '.2f'):>8}")
    print(f"\nEstimated total so far (Fable's share): ${total:.2f}")

## Step 1: a short check (5 rounds, about \$4)

**API errors** and **Truncated answers** should both be 0. If not, stop and send the output to Claude.

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT} --max-rounds 5
report("alle_kjenner_maalet")

## Step 2: everyone knows the goal (full game)

Haiku got stuck after 32 rounds with these cards. Fable solved it in round 20 on 10 November.

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("alle_kjenner_maalet")

## Step 3: the original exercise – only A knows the goal (full game)

Haiku got stuck after 18 rounds; the goal never left A. Fable solved it in round 31 on 10 November.

In [ ]:
path = prepare("grunnbetingelse")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("grunnbetingelse")

## Step 4: the procedures (only if Fable got stuck in step 2)

**4a – the plain procedure.**

In [ ]:
path = prepare("prosedyre")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("prosedyre")

**4b – the answer key's method as concrete rules.** Haiku needed this version to solve the task, in round 39.

In [ ]:
path = prepare("prosedyre_fasit_regler")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("prosedyre_fasit_regler")

## Step 5: a second card deal

Steps 2 and 3 again with **card deal 2**, where the answer key needs 16 rounds.

**5a – everyone knows the goal.**

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed 2 --concurrent {CONCURRENT}
report("alle_kjenner_maalet", seed=2)

**5b – only A knows the goal.**

In [ ]:
path = prepare("grunnbetingelse")
!python run.py {path} --model {MODEL} --seed 2 --concurrent {CONCURRENT}
report("grunnbetingelse", seed=2)

## Step 6: mixed organisations

Until now all seven agents have been copies of the same model. Here Fable works with Claude Haiku. Everyone knows the goal and the cards are deal 1, so the results compare directly with step 2.

**6a – Fable leaders (A, B, C), Haiku employees (D–G).**

In [ ]:
path = prepare_mixed("mixed_fable_leaders", leaders=MODEL, employees=SMALL)
!python run.py {path} --seed {SEED} --concurrent {CONCURRENT}
report("mixed_fable_leaders")

**6b – Haiku leaders, Fable employees.**

In [ ]:
path = prepare_mixed("mixed_fable_employees", leaders=SMALL, employees=MODEL)
!python run.py {path} --seed {SEED} --concurrent {CONCURRENT}
report("mixed_fable_employees")

## All Fable games so far

In [ ]:
fable_table()

## Download the logs

Unpack the zip file on your own computer and open the `.json` files in `replay.html` – or send the zip file to Claude.

In [ ]:
import shutil
shutil.make_archive("/content/fable_hierarchy_logs", "zip", "logger")
files.download("/content/fable_hierarchy_logs.zip")